**Setup:**

In [1]:
# Install BCFL eval for its dependencies (the pip package itself won't be used);
!pip install -q --upgrade vllm "bfcl-eval" rich requests "hf-transfer"

**Config (paths, models, flags):**

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, subprocess, re, shutil

# Config & directories;
WORKDIR = Path(".").resolve()
BFCL_ROOT = WORKDIR / "bfcl" / "bfcl_eval"
ASSERTIONS_JSONL = WORKDIR / "assertions.jsonl"
MULTI_TURN_BASE = BFCL_ROOT / "data" / "BFCL_v4_multi_turn_base.json"
LOGS_DIR = WORKDIR / "logs"; LOGS_DIR.mkdir(exist_ok=True)
BACKUP_DIR = WORKDIR / "bfcl_data_backups"; BACKUP_DIR.mkdir(exist_ok=True)
MODEL_IDS = [ # These are models that fit into a 48GB GPU
    "Qwen/Qwen3-8B-FC",
    "Qwen/Qwen3-14B-FC",
    "Salesforce/xLAM-2-3b-fc-r",
    "Salesforce/Llama-xLAM-2-8b-fc-r",
    "BitAgent/BitAgent-Bounty-8B",
    "Team-ACE/ToolACE-2-8B",
    "watt-ai/watt-tool-8B"
]
GEN_FLAGS = ["--backend","vllm","--num-gpus","1","--gpu-memory-utilization","0.95","--test-category","multi_turn_base"]
EVAL_FLAGS = ["--test-category","multi_turn_base"]
ARCHIVE_DIR  = WORKDIR / "archive"; ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR = WORKDIR / "result" # BFCL writes predictions here;
SCORES_DIR    = WORKDIR / "score" # BFCL writes scores here;

ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# Make local bfcl importable in *this* kernel & subprocesses;
if str(BFCL_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(BFCL_ROOT.parent))
os.environ.setdefault("BFCL_PROJECT_ROOT", str(WORKDIR))
os.environ.setdefault("BFCL_DATA_DIR", str(BFCL_ROOT / "data"))

def _env_local():
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{str(BFCL_ROOT.parent)}:{env.get('PYTHONPATH','')}".rstrip(":")
    env.setdefault("BFCL_PROJECT_ROOT", str(WORKDIR))
    env.setdefault("BFCL_DATA_DIR", str(BFCL_ROOT / "data"))
    return env

# Patch ONLY multi_turn_base JSONL (fast path: `question[0][0]['content']`);
def _backup(p: Path):
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    dst = BACKUP_DIR / f"{p.stem}.{ts}.bak{p.suffix}"
    dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
def _load_assertions(jsonl: Path):
    amap = {}
    with jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line)
                if o.get("id"): amap[o["id"]] = {"m": o.get("modified_first_prompt"), "a": o.get("assertion")}
    return amap

def patch_multi_turn_base():
    assert MULTI_TURN_BASE.exists(), f"Not found: {MULTI_TURN_BASE}"
    amap = _load_assertions(ASSERTIONS_JSONL)
    _backup(MULTI_TURN_BASE)
    tmp = MULTI_TURN_BASE.with_suffix(".tmp.jsonl")
    matched = modified = 0
    with MULTI_TURN_BASE.open("r", encoding="utf-8") as src, tmp.open("w", encoding="utf-8") as dst:
        for raw in src:
            if not raw.strip(): dst.write("\n"); continue
            obj = json.loads(raw)
            if obj.get("id") in amap:
                matched += 1
                q = obj.get("question", [])
                try:
                    node = q[0][0] # Fixed structure;
                    if isinstance(node, dict) and node.get("role") == "user":
                        orig = node.get("content","")
                        inj = amap[obj["id"]]["m"]
                        if isinstance(inj,str) and inj.strip():
                            node["content"] = inj; modified += 1
                        else:
                            fb = amap[obj["id"]]["a"]
                            if isinstance(fb,str) and fb.strip():
                                sep = "" if isinstance(orig,str) and orig.endswith(("\n"," ")) else "\n\n"
                                node["content"] = f"{orig}{sep}{fb}"
                                modified += 1
                except Exception:
                    pass
            dst.write(json.dumps(obj, ensure_ascii=False) + "\n")
    tmp.replace(MULTI_TURN_BASE)
    print(f"Patched multi_turn_base.jsonl  matched={matched}  modified={modified}")

# Minimal BFCL runner, running from our local module (not pip);
def _safe(s): return re.sub(r"[^A-Za-z0-9_.-]+","_",s)
def _run_bfcl(subcmd, model, flags):
    for mod, head in (("bfcl_eval.cli",[subcmd]), ("bfcl_eval.__main__", [subcmd])):
        try:
            __import__(mod)
            cmd = [sys.executable, "-m", mod] + head + ["--model", model] + flags
            log = LOGS_DIR / f"bfcl_{subcmd}_{_safe(model)}_{datetime.now().strftime('%Y%m%d-%H%M%S')}.log"
            with log.open("w", encoding="utf-8") as lf:
                # tiny sanity prelude inside the subprocess log
                sanity = "import bfcl_eval,os;print('bfcl_eval.__file__=',bfcl_eval.__file__);print('BFCL_DATA_DIR=',os.environ.get('BFCL_DATA_DIR'))"
                subprocess.run([sys.executable,"-c",sanity], cwd=str(BFCL_ROOT), env=_env_local(), stdout=lf, stderr=subprocess.STDOUT)
                lf.write(f"\n=== bfcl {subcmd} ===\n")
                proc = subprocess.Popen(cmd, cwd=str(BFCL_ROOT), env=_env_local(), stdout=lf, stderr=subprocess.STDOUT)
                rc = proc.wait()
            print(f"{subcmd} [{model}] → rc={rc}  log={log.name}")
            return rc
        except Exception:
            continue
    raise RuntimeError("Local BFCL entrypoint not found (bfcl_eval.cli / __main__).")

print("Kernel using local `bfcl_eval` at import time:")
import importlib
if "bfcl_eval" in sys.modules:  # purge any previously-imported pip copy
    del sys.modules["bfcl_eval"]
importlib.invalidate_caches()
import bfcl_eval
print("  ", bfcl_eval.__file__)
assert (BFCL_ROOT / "__init__.py").exists(), "bfcl_eval local package missing __init__.py"

In [ ]:
def _zip_tree(src_dir: Path, zip_path: Path):
    """Zip src_dir into zip_path; return True if something was archived."""
    if not src_dir.exists() or not any(src_dir.rglob("*")):
        return False
    base = zip_path.with_suffix("") # shutil.make_archive wants base name without .zip;
    shutil.make_archive(str(base), "zip", root_dir=str(src_dir))
    return True
def _archive_and_clear_shared(model_name: str):
    """Zip result/ and score/ to archive/, then clear both directories."""
    key = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_name)
    ts  = datetime.now().strftime("%Y%m%d-%H%M%S")
    res_zip = ARCHIVE_DIR / f"{key}_{ts}_result.zip"
    sco_zip = ARCHIVE_DIR / f"{key}_{ts}_score.zip"

    ok_res = _zip_tree(ARTIFACTS_DIR, res_zip)
    ok_sco = _zip_tree(SCORES_DIR,   sco_zip)
    print(f"[archive] result → {res_zip.name}" if ok_res else "[archive] no result/ to archive")
    print(f"[archive] score  → {sco_zip.name}" if ok_sco else "[archive] no score/ to archive")

    # Clear both (safe even if empty)
    for d in (ARTIFACTS_DIR, SCORES_DIR):
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
        print(f"[clean] cleared {d}")

def _clear_shared_before_generate():
    """Defensive pre-clean (handles aborted previous runs)."""
    for d in (ARTIFACTS_DIR, SCORES_DIR):
        if d.exists() and any(d.rglob("*")):
            shutil.rmtree(d)
            print(f"[pre-clean] cleared {d}")
        d.mkdir(parents=True, exist_ok=True)

# Patch dataset with assertions (comment out if not evaluating on assertions);
patch_multi_turn_base()

# Begin running BFCL on model IDs;
results = []
for i, m in enumerate(MODEL_IDS, 1):
    print(f"\n=== [{i}/{len(MODEL_IDS)}] GENERATE start: {m} ===")
    _clear_shared_before_generate() # Ensure no leftovers;
    rc_g = _run_bfcl("generate", m, GEN_FLAGS) # BFCL spins up ONE vLLM for this model;
    print(f"\n=== [{i}/{len(MODEL_IDS)}] EVALUATE start: {m} ===")
    rc_e = _run_bfcl("evaluate", m, EVAL_FLAGS)
    # Archive outputs for this model, then clear shared dirs;
    _archive_and_clear_shared(m)

    results.append((m, rc_g, rc_e))
    print(f"--- Completed: {m} (generate={rc_g}, evaluate={rc_e}). Moving to next model... ---")

print("\nSummary:")
for m, g, e in results:
    print(f"  {m}: generate={g}, evaluate={e}")
print("Archives in:", WORKDIR / "archive", "| Logs in:", LOGS_DIR)